In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
np.random.normal(0, 1, (10, 3))

array([[-0.5649345 , -0.66459974,  1.3157613 ],
       [ 1.36759913, -0.65765207,  0.23514372],
       [-0.17359215,  1.18837042, -1.2215716 ],
       [-0.39846206, -0.07121298,  0.84938735],
       [-0.37166915, -0.34529254, -0.78923415],
       [ 1.61027222,  0.66863774, -0.70020673],
       [-0.94007018, -0.69775067, -1.47216013],
       [ 0.03789733,  0.15624505, -0.52502429],
       [ 0.93064103, -0.17359604, -1.82690964],
       [-1.13514871,  0.36561859, -2.35275526]])

In [27]:
rng = np.random.default_rng(9)

In [28]:
normal_dist = rng.normal(0, 1, (4, 3))
normal_dist

array([[-0.80283694,  0.24284991, -1.65634543],
       [ 0.65610488,  1.14345302, -0.452611  ],
       [ 0.43048575,  0.25093257, -0.39435206],
       [-0.86240487, -2.03255243,  1.41042348]])

In [29]:
normal_dist.cumsum(axis=1)

array([[-0.80283694, -0.55998703, -2.21633246],
       [ 0.65610488,  1.7995579 ,  1.3469469 ],
       [ 0.43048575,  0.68141831,  0.28706626],
       [-0.86240487, -2.89495729, -1.48453381]])

In [30]:
np.zeros((4, 1))

array([[0.],
       [0.],
       [0.],
       [0.]])

In [31]:
np.concatenate([np.zeros((4, 1)), normal_dist.cumsum(axis=1)], axis=1) 

array([[ 0.        , -0.80283694, -0.55998703, -2.21633246],
       [ 0.        ,  0.65610488,  1.7995579 ,  1.3469469 ],
       [ 0.        ,  0.43048575,  0.68141831,  0.28706626],
       [ 0.        , -0.86240487, -2.89495729, -1.48453381]])

In [ ]:
# for stock price we typically use GBM to model it
# GBMParams: 1. mu, 2, sigma
# GBM log return mean: ( mu - sigma ** 2 / 2 ) * dt
# GBM log return stdev: sigma * sqrt (dt)
# log return = log return mean + log return stdev * N(0, 1)

In [158]:
def generate_gbm_log_return_paths(
    mu: float, 
    sigma: float, 
    num_path: int, 
    num_period: int, 
    dt: float, 
    rng: np.random.Generator):
    gbm_log_return_mean = (mu - sigma ** 2 / 2) * dt
    gbm_log_return_stdev = sigma * np.sqrt(dt)
    print(f'gbm log return mean: {gbm_log_return_mean}, gbm_log_return_stdev: {gbm_log_return_stdev}')
    log_returns = rng.normal(gbm_log_return_mean, gbm_log_return_stdev, (num_path, num_period))
    return log_returns

In [159]:
log_ret_4_3 = generate_gbm_log_return_paths(0, 1, 4, 3, 1.0, np.random.default_rng(9))
log_ret_4_3

gbm log return mean: -0.5, gbm_log_return_stdev: 1.0


array([[-1.30283694, -0.25715009, -2.15634543],
       [ 0.15610488,  0.64345302, -0.952611  ],
       [-0.06951425, -0.24906743, -0.89435206],
       [-1.36240487, -2.53255243,  0.91042348]])

In [45]:
np.cumsum(log_ret_4_3, axis=1)

array([[-1.30283694, -1.55998703, -3.71633246],
       [ 0.15610488,  0.7995579 , -0.1530531 ],
       [-0.06951425, -0.31858169, -1.21293374],
       [-1.36240487, -3.89495729, -2.98453381]])

In [49]:
log_ret_paths = np.concatenate([np.zeros((4, 1)), np.cumsum(log_ret_4_3, axis=1)], axis=1)
log_ret_paths

array([[ 0.        , -1.30283694, -1.55998703, -3.71633246],
       [ 0.        ,  0.15610488,  0.7995579 , -0.1530531 ],
       [ 0.        , -0.06951425, -0.31858169, -1.21293374],
       [ 0.        , -1.36240487, -3.89495729, -2.98453381]])

In [52]:
init_px = 100
np.exp(log_ret_paths) * init_px

array([[100.        ,  27.17597334,  21.01387969,   2.43230102],
       [100.        , 116.89487932, 222.45572349,  85.8084154 ],
       [100.        ,  93.28468361,  72.71796756,  29.73237279],
       [100.        ,  25.60442839,   2.03442433,   5.05630701]])

In [193]:
def get_prices_paths(log_return_paths: np.ndarray, init_px: float):
    if np.ndim(log_return_paths) != 2:
        raise RuntimeError("need 2 dimensional log return paths")
    num_path, num_period = log_return_paths.shape
    return_paths = np.concatenate([np.zeros((num_path, 1)), np.cumsum(log_return_paths, axis=1)], axis=1)
    return init_px * np.exp(return_paths)

In [69]:
price_paths = get_prices_paths(log_ret_4_3, 100)
price_paths

array([[100.        ,  27.17597334,  21.01387969,   2.43230102],
       [100.        , 116.89487932, 222.45572349,  85.8084154 ],
       [100.        ,  93.28468361,  72.71796756,  29.73237279],
       [100.        ,  25.60442839,   2.03442433,   5.05630701]])

In [123]:
price_paths[:, [2, 3]]

array([[ 21.01387969,   2.43230102],
       [222.45572349,  85.8084154 ],
       [ 72.71796756,  29.73237279],
       [  2.03442433,   5.05630701]])

In [73]:
price_paths[:, 0:3]

array([[100.        ,  27.17597334,  21.01387969],
       [100.        , 116.89487932, 222.45572349],
       [100.        ,  93.28468361,  72.71796756],
       [100.        ,  25.60442839,   2.03442433]])

In [96]:
price_paths[:, 2][:, None] # do no use reshape

array([[ 21.01387969],
       [222.45572349],
       [ 72.71796756],
       [  2.03442433]])

In [150]:
K = 100
price_paths[:, 2] - K, np.maximum(price_paths[:, 2] - K, 0)

(array([-78.98612031, 122.45572349, -27.28203244, -97.96557567]),
 array([  0.        , 122.45572349,   0.        ,   0.        ]))

In [127]:
Ks = np.array([95, 100, 105])
Ks.shape, Ks

((3,), array([ 95, 100, 105]))

In [128]:
price_paths[:, 2][:, None] - Ks

array([[ -73.98612031,  -78.98612031,  -83.98612031],
       [ 127.45572349,  122.45572349,  117.45572349],
       [ -22.28203244,  -27.28203244,  -32.28203244],
       [ -92.96557567,  -97.96557567, -102.96557567]])

In [129]:
price_paths[:, 2][:, None] - Ks[None, :]

array([[ -73.98612031,  -78.98612031,  -83.98612031],
       [ 127.45572349,  122.45572349,  117.45572349],
       [ -22.28203244,  -27.28203244,  -32.28203244],
       [ -92.96557567,  -97.96557567, -102.96557567]])

In [137]:
price_paths_2_and_3 = price_paths[:, [2, 3]]
price_paths_2_and_3

array([[ 21.01387969,   2.43230102],
       [222.45572349,  85.8084154 ],
       [ 72.71796756,  29.73237279],
       [  2.03442433,   5.05630701]])

In [138]:
price_paths_2_and_3- 100, price_paths_2_and_3-105

(array([[-78.98612031, -97.56769898],
        [122.45572349, -14.1915846 ],
        [-27.28203244, -70.26762721],
        [-97.96557567, -94.94369299]]),
 array([[ -83.98612031, -102.56769898],
        [ 117.45572349,  -19.1915846 ],
        [ -32.28203244,  -75.26762721],
        [-102.96557567,  -99.94369299]]))

In [143]:
price_paths_2_and_3[:, :, None].shape, Ks.shape

((4, 2, 1), (3,))

In [144]:
price_paths_2_and_3[:, :, None] - Ks

array([[[ -73.98612031,  -78.98612031,  -83.98612031],
        [ -92.56769898,  -97.56769898, -102.56769898]],

       [[ 127.45572349,  122.45572349,  117.45572349],
        [  -9.1915846 ,  -14.1915846 ,  -19.1915846 ]],

       [[ -22.28203244,  -27.28203244,  -32.28203244],
        [ -65.26762721,  -70.26762721,  -75.26762721]],

       [[ -92.96557567,  -97.96557567, -102.96557567],
        [ -89.94369299,  -94.94369299,  -99.94369299]]])

In [151]:
np.mean(np.maximum(price_paths_2_and_3[:, :, None] - Ks, 0), axis=0)

array([[31.86393087, 30.61393087, 29.36393087],
       [ 0.        ,  0.        ,  0.        ]])

In [156]:
price_paths, log_ret_paths

(array([[100.        ,  27.17597334,  21.01387969,   2.43230102],
        [100.        , 116.89487932, 222.45572349,  85.8084154 ],
        [100.        ,  93.28468361,  72.71796756,  29.73237279],
        [100.        ,  25.60442839,   2.03442433,   5.05630701]]),
 array([[ 0.        , -1.30283694, -1.55998703, -3.71633246],
        [ 0.        ,  0.15610488,  0.7995579 , -0.1530531 ],
        [ 0.        , -0.06951425, -0.31858169, -1.21293374],
        [ 0.        , -1.36240487, -3.89495729, -2.98453381]]))

In [161]:
np.array([1, 2, 3])[:, None] - np.array([2, 3])

array([[-1, -2],
       [ 0, -1],
       [ 1,  0]])

In [201]:
def evaluate_price(price_paths, strikes, matury_indices):
    payoff = np.mean(np.maximum(price_paths[:, matury_indices][:, :, None] - strikes, 0), axis=0)
    print(f'payoff in the maturies:\n {payoff}')
    return payoff

In [166]:
payoff = evaluate_price(price_paths, np.array([95, 100, 105]), [2, 3])

payoff in the maturies: [[31.86393087 30.61393087 29.36393087]
 [ 0.          0.          0.        ]]


In [167]:
payoff

array([[31.86393087, 30.61393087, 29.36393087],
       [ 0.        ,  0.        ,  0.        ]])

In [168]:
payoff.shape


(2, 3)

In [170]:
coeff = np.array([10, 100])
coeff[:, None]

array([[ 10],
       [100]])

In [184]:
payoff / coeff[:, None] # vectorization used to calculate present values

array([[3.18639309, 3.06139309, 2.93639309],
       [0.        , 0.        , 0.        ]])

In [176]:
r = 0.03
mu = r
sigma = 0.2
dt = 1/252
T = 1

In [186]:
px = 100
n_path = 64
n_period = 120
strikes = np.arange(90, 111)
maturities = [30, 60, 90, 120]

In [195]:
log_returns_long = generate_gbm_log_return_paths(mu, sigma, n_path, n_period, dt, np.random.default_rng(10))
log_returns_long

gbm log return mean: 3.968253968253966e-05, gbm_log_return_stdev: 0.012598815766974242


array([[-0.01386108, -0.00909477, -0.00981014, ..., -0.00892897,
         0.02085359, -0.002116  ],
       [ 0.02050685, -0.00914519, -0.02218449, ...,  0.0023959 ,
         0.00105783, -0.02064518],
       [ 0.00468222, -0.00869439,  0.00499953, ...,  0.00093598,
         0.00321939, -0.02077635],
       ...,
       [-0.00608358,  0.00104866,  0.00469084, ...,  0.00117721,
        -0.00923436, -0.01146942],
       [ 0.00317363,  0.01414605, -0.02238215, ..., -0.01175076,
        -0.00384755, -0.00779472],
       [ 0.02099752,  0.01118531, -0.03042884, ...,  0.0054283 ,
        -0.00966954, -0.0183821 ]], shape=(64, 120))

In [195]:
log_returns_long = generate_gbm_log_return_paths(mu, sigma, n_path, n_period, dt, np.random.default_rng(10))
log_returns_long

gbm log return mean: 3.968253968253966e-05, gbm_log_return_stdev: 0.012598815766974242


array([[-0.01386108, -0.00909477, -0.00981014, ..., -0.00892897,
         0.02085359, -0.002116  ],
       [ 0.02050685, -0.00914519, -0.02218449, ...,  0.0023959 ,
         0.00105783, -0.02064518],
       [ 0.00468222, -0.00869439,  0.00499953, ...,  0.00093598,
         0.00321939, -0.02077635],
       ...,
       [-0.00608358,  0.00104866,  0.00469084, ...,  0.00117721,
        -0.00923436, -0.01146942],
       [ 0.00317363,  0.01414605, -0.02238215, ..., -0.01175076,
        -0.00384755, -0.00779472],
       [ 0.02099752,  0.01118531, -0.03042884, ...,  0.0054283 ,
        -0.00966954, -0.0183821 ]], shape=(64, 120))

In [197]:
prices_long = get_prices_paths(log_returns_long, px)
prices_long

array([[100.        ,  98.62345471,  97.73056361, ...,  79.87404338,
         81.55719257,  81.38479982],
       [100.        , 102.07185559, 101.14264429, ...,  68.05749779,
         68.12952886,  66.73740232],
       [100.        , 100.46931996,  99.59958702, ...,  94.44120845,
         94.74574181,  92.79757906],
       ...,
       [100.        ,  99.39348836,  99.49777326, ...,  83.80613949,
         83.03580564,  82.0888735 ],
       [100.        , 100.31786691, 101.74705321, ...,  95.99821886,
         95.62957059,  94.88706287],
       [100.        , 102.12195217, 103.2706297 , ..., 125.44834181,
        124.24115963, 121.97820922]], shape=(64, 121))

In [202]:
payoff_long = evaluate_price(prices_long, strikes, maturities)

payoff in the maturies:
 [[11.03608325 10.10713929  9.19408244  8.30936113  7.43436113  6.58799924
   5.77766756  5.01955759  4.30551513  3.66185019  3.08969185  2.53614133
   2.03573038  1.61525589  1.23858274  0.93002836  0.6978824   0.5260074
   0.38083377  0.25399175  0.15559411]
 [10.18030103  9.34201408  8.53614749  7.74694151  6.98947634  6.2835927
   5.59872827  4.98292444  4.40211183  3.8671718   3.37639339  2.919002
   2.49624602  2.11250589  1.76904934  1.46349568  1.1922923   0.96554718
   0.74776464  0.58553065  0.46910842]
 [10.55383037  9.77869158  9.02869158  8.29566183  7.60272126  6.92756347
   6.28446456  5.68510998  5.13426105  4.65798954  4.20762318  3.80478421
   3.44409326  3.10705133  2.80929505  2.53411817  2.28573426  2.05135926
   1.81698426  1.59530068  1.3981984 ]
 [10.00407532  9.30697116  8.62463471  7.99649586  7.41042412  6.84412541
   6.31437446  5.84065144  5.38752644  4.94807276  4.52016459  4.11391459
   3.73846734  3.41317769  3.11630269  2.8194276